# Network Analysis Sandbox
This notebook loads the stock similarity matrix and builds a network using the logic from `main.py`. Use the cells below to explore the graph structure and run custom analysis.

In [ ]:
import sys
import os
import networkx as nx
import pandas as pd
from pathlib import Path

# Add project root to sys.path to allow importing main.py
sys.path.append(os.path.abspath('.'))

# Import the existing build_network function
from main import build_network

# Configuration
output_dir = Path("output")
similarity_file = output_dir / "ticker_similarities.csv"

# Load and process the similarity matrix
if not similarity_file.exists():
    print(f"Error: {similarity_file} not found. Please run data_processor.py first.")
else:
    # Load the similarity matrix
    sim_df = pd.read_csv(similarity_file, index_col=0)

    # Calculate simple cosine distance: d = 1 - s
    cosine_dist = 1.0 - sim_df.values
    dist_df = pd.DataFrame(cosine_dist, index=sim_df.index, columns=sim_df.columns)

    # Build the network using the shared logic
    G = build_network(dist_df)
    print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

### Basic Visualization and Top Closest Pairs
Identify the stocks that have the lowest cosine distance (and thus highest similarity).

In [ ]:
# Display Top 10 Closest Ticker Pairs
print("\nTop 10 Closest Ticker Pairs (by Cosine Distance):")
edges = sorted(G.edges(data=True), key=lambda x: x[2].get("weight", 1.0))

count = 0
for u, v, d in edges:
    if count >= 10:
        break
    print(f"{u} <-> {v}: distance = {d['weight']:.4f}")
    count += 1

count = 0
for u, v, d in edges:
    if count >= 10:
        break
    print(f"{u} <-> {v}: distance = {d['weight']:.4f}")
    count += 1


In [ ]:
# Calculate hubs in the Minimum Spanning Tree (MST)
# T is the MST calculated in the previous cell
T = nx.minimum_spanning_tree(G, weight="weight")

# Get degrees for all nodes in the MST
mst_degrees = dict(T.degree())

# Sort nodes by degree in descending order
top_mst_hubs = sorted(mst_degrees.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 Hub Nodes in the MST (Highest Degree):")
for node, degree in top_mst_hubs:
    print(f"{node}: degree = {degree}")

# Optional: Highlight these hubs in a printout of their neighbors
print("\nConnections for the top MST hub:")
top_hub = top_mst_hubs[0][0]
neighbors = list(T.neighbors(top_hub))
print(f"{top_hub} is connected to: {', '.join(neighbors)}")
